## 30_export_models  (OLD workspace) — Model Artifacts Migration step 1 of 3

Video practice (frame 29): a CSV list drives the exporter.
  1. Build model_list.csv for the 5 practice models
  2. Download each model's artifacts (MLmodel, pkl, ...)
  3. Score 5 fixed rows FROM THE TRAINING TABLE -> parity_reference.csv
     + parity_features.json  (for post-migration parity)
  4. Zip everything -> models_export.zip -> publish for download
Tested on DBR 12.2 (MLflow 2.x) and newer; the download layer normalizes
both known artifact layouts so 50_register_models always finds MLmodel.


In [ ]:

%run ./lib/config_lib


In [ ]:

import os, shutil
import numpy as np
import pandas as pd
import mlflow
from mlflow.tracking import MlflowClient

MODELS = [
    "practice_linear_regression",
    "practice_decision_tree",
    "practice_random_forest",
    "practice_xgboost",
    "practice_sklearn_pipeline",
]

CFG = setup_config(config_path="", run_date="all", schema="practice")

EXPORT_DIR = "/tmp/migration_export"
shutil.rmtree(EXPORT_DIR, ignore_errors=True)
os.makedirs(EXPORT_DIR, exist_ok=True)
client = MlflowClient()


In [ ]:

# Step 1 — the CSV manifest (video: "create the model list CSV before export")
rows = []
for simple in MODELS:
    reg = CFG.model(simple)  # Free Edition UC registry: <current_catalog>.default.<name>
    versions = client.search_model_versions(f"name = '{reg}'")
    assert versions, f"No registered model named '{reg}' — run the training notebook first"
    latest = max(versions, key=lambda v: int(v.version))
    rows.append({
        "name": reg,
        "simple_name": simple,
        "version": int(latest.version),
        "run_id": latest.run_id,
    })

manifest = pd.DataFrame(rows)
manifest.to_csv(os.path.join(EXPORT_DIR, "model_list.csv"), index=False)
display(manifest)


In [ ]:

# Step 2 — download each model's artifact folder into a per-model directory.
# Layouts differ across MLflow/runtime versions:
#   old: mlflow.artifacts.download_artifacts(models:/...) -> <stage>/model/MLmodel
#   new: same call -> <stage>/MLmodel (already flat)
# Normalize by locating MLmodel anywhere under the staging dir, then moving
# that folder to EXPORT_DIR/<simple_name>. This is what 50_register_models
# expects (<simple_name>/MLmodel), and it fixes the classic nesting bug.
def download_model_artifacts(name, version, run_id, simple, stage_root):
    stage = os.path.join(stage_root, simple)
    shutil.rmtree(stage, ignore_errors=True)
    try:
        mlflow.artifacts.download_artifacts(f"models:/{name}/{version}", dst_path=stage)
    except AttributeError:
        # DBR 12.2 fallback path (older MLflow API)
        client.download_artifacts(run_id, "model", dst_path=stage)

    found = None
    for root, _dirs, files in os.walk(stage):
        if "MLmodel" in files:
            found = root
            break
    assert found, f"MLmodel not found under {stage} for {name} v{version}"

    dest = os.path.join(EXPORT_DIR, simple)
    shutil.rmtree(dest, ignore_errors=True)
    shutil.move(found, dest)
    shutil.rmtree(stage, ignore_errors=True)
    return dest


STAGE_ROOT = os.path.join(EXPORT_DIR, "_staging")
for r in rows:
    dest = download_model_artifacts(
        r["name"], r["version"], r["run_id"], r["simple_name"], STAGE_ROOT
    )
    files = os.listdir(dest)
    assert "MLmodel" in files, f"MLmodel missing for {r['name']}"
    print(f"OK  {r['simple_name']}  ({len(files)} files)  <- models:/{r['name']}/{r['version']}")


In [ ]:

# Step 3 — parity reference: score 5 fixed rows FROM THE TRAINING TABLE with the
# SOURCE models. The exact feature rows go into parity_features.json so
# 60_verify_models reconstructs the same matrix without the old workspace.
import json

# CFG defined in the setup cell above (schema.table names; no catalog for tables)
X_parity, parity_ids = parity_rows(CFG.table("practice_customers"))
with open(os.path.join(EXPORT_DIR, "parity_features.json"), "w") as f:
    json.dump({"features": FEATURES, "rows": X_parity.tolist(), "customer_ids": parity_ids}, f, indent=2)

parity_records = []
for r in rows:
    m = mlflow.pyfunc.load_model(f"models:/{r['name']}/{r['version']}")
    preds = np.asarray(m.predict(X_parity), dtype=float).ravel()
    rec = {"simple_name": r["simple_name"], "version": r["version"]}
    for i, p in enumerate(preds):
        rec[f"pred_{i}"] = p
    parity_records.append(rec)

pd.DataFrame(parity_records).to_csv(os.path.join(EXPORT_DIR, "parity_reference.csv"), index=False)
print(f"Parity reference saved (5 training-table rows per model; ids={parity_ids}).")


In [ ]:

# Step 4 — zip and publish for download to your laptop.
ZIP_LOCAL = shutil.make_archive("/tmp/models_export", "zip", EXPORT_DIR)
print(f"Zip ready: {ZIP_LOCAL}  ({os.path.getsize(ZIP_LOCAL)/1024:.1f} KB)")

user = dbutils.notebook.getContext().userName().get()
fname = os.path.basename(ZIP_LOCAL)
try:
    ws_dir = f"/Workspace/Users/{user}/migration_export"
    os.makedirs(ws_dir, exist_ok=True)
    shutil.copy(ZIP_LOCAL, ws_dir)
    print(f"\nDOWNLOAD from the workspace file browser:\n  {ws_dir}/{fname}\n(right-click -> Download)")
except Exception as e:
    print(f"Workspace copy failed ({e}); falling back to DBFS FileStore...")
    dbutils.fs.cp(f"file:{ZIP_LOCAL}", f"dbfs:/FileStore/migration_export/{fname}")
    print(f"Download from: https://<your-old-workspace-url>/files/migration_export/{fname}")
